# 06. Spectroscopy (IR / UV-Vis) & Powder X-Ray Diffraction

`xkcd-matchem` computes simulated powder XRD using pure-NumPy Laue reciprocal lattice generation and **Cromer-Mann (LA-3816)** atomic scattering form factors (plus the exact analytical hydrogen $1s$ factor) at $\text{Cu K}\alpha_1$ (`1.540598` Å per CODATA 2022).

:::{note}
Remember our design boundary: xkcd-style wiggly plots are ideal for visualizing peak patterns cleanly in lectures, notebooks, and summaries—while archival peak fitting and Rietveld refinements always belong in crisp non-wiggly plots.
:::

## Basic Usage

Plot vibrational IR spectra (`h2o_gaussian.log`), TDDFT UV-Vis transitions (`benzene_orca.out`), and crystal powder XRD (`si_diamond.cif`) with clean default presets:

In [ ]:
import matplotlib.pyplot as plt
import xkcd_matchem as xm

h2o = xm.load("h2o_gaussian.log")
benz = xm.load("benzene_orca.out")

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.3))
    xm.plot_spectrum(h2o.metadata["ir_spectrum"], ax=axes[0], title="H2O IR (Gaussian)")
    xm.plot_spectrum(benz.metadata["uvvis_spectrum"], ax=axes[1], title="Benzene UV-Vis (ORCA)")
    xm.plot_xrd("si_diamond.cif", two_theta_range=(20.0, 85.0), ax=axes[2], title="Si Cu Ka1 Powder XRD")
    fig.tight_layout()

## Advanced Usage

Generate custom pseudo-Voigt broadening with `xm.broaden_peaks(...)`, inspect peak assignments on `compute_xrd(...)`, and switch radiation wavelength or theme colors cleanly:

In [ ]:
import numpy as np

nacl_spec = xm.load("nacl.cif")
xrd_spec = xm.compute_xrd(nacl_spec, two_theta_range=(15.0, 75.0), fwhm=0.55)
print("Top 3 reflections:", [(round(p.position, 2), round(p.intensity, 1), p.label) for p in sorted(xrd_spec.peaks, key=lambda k: k.intensity, reverse=True)[:3]])

custom_grid = np.linspace(15.0, 75.0, 800)
pv_curve = xm.broaden_peaks(
    [p.position for p in xrd_spec.peaks],
    [p.intensity for p in xrd_spec.peaks],
    custom_grid,
    fwhm=0.8,
    profile="pseudo_voigt",
)
pv_spec = xm.Spectrum(
    x=custom_grid,
    y=pv_curve / max(pv_curve.max(), 1e-6) * 100.0,
    kind="xrd",
    x_label="2theta (deg)",
    y_label="Normalized Intensity (%)",
    peaks=xrd_spec.peaks,
)
ax = xm.plot_xrd(pv_spec, two_theta_range=(15.0, 75.0), label_peaks=4, color="#882255", title="Custom Pseudo-Voigt NaCl XRD")